In [1]:
import pandas as pd

# Load the original hospital dataset
df = pd.read_csv("HospitalOPD-noname.csv")

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Dataset loaded successfully!
Number of rows: 62422
Number of columns: 25


In [2]:
# Remove the completely empty column, if it exists
df_clean = df.drop(columns=["Unnamed: 2"], errors="ignore").copy()

# Convert activity timestamps into datetime format
timestamp_columns = [
    "appointment_start_datetime",
    "prescription_start_datetime",
    "prescription_end_datetime",
    "lab_start_datetime",
    "lab_end_datetime",
    "imaging_start_datetime",
    "imaging_end_datetime",
    "pharmacy_start_datetime",
    "pharmacy_end_datetime"
]

for col in timestamp_columns:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(
            df_clean[col], errors="coerce"
        )

print("Data preparation completed!")
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

Data preparation completed!
Rows: 62422
Columns: 24


In [3]:
# Day 4: Build the hospital event log

# Create one appointment event per appointment ID
appointment_events = (
    df_clean[
        ["appointment_id", "appointment_start_datetime"]
    ]
    .dropna(subset=["appointment_id", "appointment_start_datetime"])
    .drop_duplicates(subset=["appointment_id"])
    .rename(columns={
        "appointment_id": "case_id",
        "appointment_start_datetime": "timestamp"
    })
    .assign(activity="Appointment")
)

event_parts = [appointment_events]

# Add hospital service-start events
service_columns = {
    "Prescription Started": "prescription_start_datetime",
    "Lab Started": "lab_start_datetime",
    "Imaging Started": "imaging_start_datetime",
    "Pharmacy Started": "pharmacy_start_datetime"
}

for activity, column in service_columns.items():
    part = (
        df_clean[["appointment_id", column]]
        .dropna(subset=["appointment_id", column])
        .rename(columns={
            "appointment_id": "case_id",
            column: "timestamp"
        })
        .assign(activity=activity)
    )

    event_parts.append(part)

# Combine all events into one table
event_log_clean = pd.concat(event_parts, ignore_index=True)

# Sort events by appointment and time
event_log_clean = (
    event_log_clean
    .sort_values(["case_id", "timestamp", "activity"])
    .reset_index(drop=True)
)

print("Event log created successfully!")
print("Total events:", len(event_log_clean))
print("Unique appointments:", event_log_clean["case_id"].nunique())
print("Activity types:", event_log_clean["activity"].nunique())

Event log created successfully!
Total events: 136176
Unique appointments: 49912
Activity types: 5


In [4]:


# Count how many times each activity occurs per appointment
activity_counts = (
    event_log_clean
    .groupby(["case_id", "activity"])
    .size()
    .reset_index(name="event_count")
)

# Find activities that occur more than once per appointment
repeated_activities = activity_counts[
    activity_counts["event_count"] > 1
].copy()

# Summarize repeated activities without displaying appointment IDs
repeated_summary = (
    repeated_activities
    .groupby("activity")
    .agg(
        appointments_with_repeats=("case_id", "nunique"),
        total_events_in_these_appointments=("event_count", "sum")
    )
    .reset_index()
    .sort_values(
        "appointments_with_repeats",
        ascending=False
    )
)

print(
    "Appointments with at least one repeated activity:",
    repeated_activities["case_id"].nunique()
)

print("\nRepeated Activity Summary:")
display(repeated_summary)

Appointments with at least one repeated activity: 4821

Repeated Activity Summary:


,activity,appointments_with_repeats,total_events_in_these_appointments
3,Prescription Started,4821,17330
1,Lab Started,4132,15791
0,Imaging Started,187,387
2,Pharmacy Started,82,276


In [5]:
# Save aggregate repeated-activity results only
repeated_summary.to_csv(
    "careflow_repeated_activity_summary.csv",
    index=False
)

print("Privacy-safe summary saved successfully!")
print("Filename: careflow_repeated_activity_summary.csv")

Privacy-safe summary saved successfully!
Filename: careflow_repeated_activity_summary.csv
